# Experiment 03 — Fine-tuning: mT5-small + QLoRA (4-bit, LoRA r=8)

**Model:** `google/mt5-small`  
**Technique:** QLoRA — 4-bit NF4 quantization + LoRA adapters  
**Goal:** First fine-tuned model. Significant ROUGE improvement over zero-shot baseline expected.  
**Columns:** `ID` | `input` | `output` | `subset`

## Install dependencies

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate rouge-score

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/health_qa/'
SAVE_DIR = '/content/drive/MyDrive/health_qa/outputs/'
CKPT_DIR = '/content/drive/MyDrive/health_qa/checkpoints/exp03/'

import os
for d in [SAVE_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

## Imports and seed

In [ ]:
import random, re, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.notebook import tqdm
from rouge_score import rouge_scorer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## Load data

In [ ]:
train_df = pd.read_csv(DATA_DIR + 'Train.csv')
val_df   = pd.read_csv(DATA_DIR + 'Val.csv')
test_df  = pd.read_csv(DATA_DIR + 'Test.csv')

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

## Parse language and country from `subset`

In [ ]:
SUBSET_MAP = {
    'Aka_Gha': ('Akan',    'Ghana'),
    'Amh_Eth': ('Amharic', 'Ethiopia'),
    'Lug_Uga': ('Luganda', 'Uganda'),
    'Swa_Ken': ('Swahili', 'Kenya'),
    'Swa_Tan': ('Swahili', 'Tanzania'),
    'Swa_Uga': ('Swahili', 'Uganda'),
    'Eng_Gha': ('English', 'Ghana'),
    'Eng_Uga': ('English', 'Uganda'),
    'Eng_Ken': ('English', 'Kenya'),
}

for df in [train_df, val_df, test_df]:
    df['language'] = df['subset'].map(lambda s: SUBSET_MAP.get(s, (s, ''))[0])
    df['country']  = df['subset'].map(lambda s: SUBSET_MAP.get(s, ('', s))[1])

print(train_df[['subset','language','country']].drop_duplicates().sort_values('subset').to_string(index=False))

## Clean text and build prompts

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'[\r\n\t]+', ' ', text)
    return re.sub(r' {2,}', ' ', text).strip()

def build_prompt(question, language, country=''):
    header = f'Language: {language}'
    if country:
        header += f' | Country: {country}'
    return f'{header}\nQuestion: {question}\nAnswer:'

for df in [train_df, val_df, test_df]:
    df['input_clean'] = df['input'].apply(clean_text)
    df['prompt_text'] = df.apply(
        lambda r: build_prompt(r['input_clean'], r['language'], r['country']), axis=1
    )

for df in [train_df, val_df]:
    df['output_clean'] = df['output'].apply(clean_text)

print(train_df['prompt_text'].iloc[0])
print()
print(train_df['output_clean'].iloc[0])

## Tokenize datasets

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME     = 'google/mt5-small'
MAX_INPUT_LEN  = 512
MAX_TARGET_LEN = 200

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    model_inputs = tokenizer(
        batch['prompt_text'], max_length=MAX_INPUT_LEN, truncation=True, padding='max_length'
    )
    labels = tokenizer(
        text_target=batch['output_clean'], max_length=MAX_TARGET_LEN, truncation=True, padding='max_length'
    )
    labels['input_ids'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in lbl]
        for lbl in labels['input_ids']
    ]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_hf  = Dataset.from_pandas(train_df[['prompt_text', 'output_clean']].reset_index(drop=True))
val_hf    = Dataset.from_pandas(val_df[['prompt_text', 'output_clean']].reset_index(drop=True))
train_tok = train_hf.map(tokenize, batched=True, remove_columns=train_hf.column_names, desc='Train')
val_tok   = val_hf.map(tokenize,   batched=True, remove_columns=val_hf.column_names,   desc='Val')

print(f'Tokenized — Train: {len(train_tok)} | Val: {len(val_tok)}')

## Load model with QLoRA (4-bit + LoRA r=8)

In [ ]:
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map='auto'
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.SEQ_2_SEQ_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Fine-tune

In [ ]:
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)

training_args = Seq2SeqTrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    predict_with_generate=True,
    generation_max_length=200,
    logging_steps=100,
    seed=SEED,
    report_to='none',
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, pad_to_multiple_of=8, label_pad_token_id=-100
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## Plot learning curves

In [ ]:
log_history = trainer.state.log_history
train_logs  = [x for x in log_history if 'loss' in x and 'eval_loss' not in x]
eval_logs   = [x for x in log_history if 'eval_loss' in x]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot([x['step'] for x in train_logs], [x['loss'] for x in train_logs],
        label='Train loss', color='steelblue')
ax.plot([x['step'] for x in eval_logs],  [x['eval_loss'] for x in eval_logs],
        label='Val loss', color='coral', marker='o')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Exp 03 — Learning curves (mT5-small + QLoRA r=8)')
ax.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR + 'exp03_learning_curve.png', dpi=150)
plt.show()

## Batch generation and ROUGE helpers

In [ ]:
_scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

def evaluate_rouge(predictions, references):
    r1s, rls = [], []
    for pred, ref in zip(predictions, references):
        s = _scorer.score(ref, pred)
        r1s.append(s['rouge1'].fmeasure)
        rls.append(s['rougeL'].fmeasure)
    return {
        'rouge1': round(sum(r1s) / len(r1s), 4),
        'rougeL': round(sum(rls) / len(rls), 4),
    }

def generate_batch(prompts, batch_size=8, max_new_tokens=200):
    model.eval()
    predictions = []
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch  = prompts[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors='pt', padding=True,
                            truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                  num_beams=4, no_repeat_ngram_size=3, early_stopping=True)
        predictions.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return predictions

## Evaluate on full validation set

In [ ]:
val_preds  = generate_batch(val_df['prompt_text'].tolist())
val_scores = evaluate_rouge(val_preds, val_df['output_clean'].tolist())

print(f"ROUGE-1 F1 : {val_scores['rouge1']}")
print(f"ROUGE-L F1 : {val_scores['rougeL']}")
print(f"Est. LB    : {0.37*val_scores['rouge1'] + 0.37*val_scores['rougeL']:.4f}")

## Per-language ROUGE breakdown

In [ ]:
rows = []
for pred, ref, lang in zip(val_preds, val_df['output_clean'], val_df['language']):
    s = _scorer.score(ref, pred)
    rows.append({'language': lang, 'rouge1': s['rouge1'].fmeasure, 'rougeL': s['rougeL'].fmeasure})

summary = (
    pd.DataFrame(rows)
    .groupby('language')
    .agg(rouge1=('rouge1', 'mean'), rougeL=('rougeL', 'mean'), count=('rouge1', 'count'))
    .round(4)
    .reset_index()
    .sort_values('rougeL', ascending=False)
)
print(summary.to_string(index=False))

## Generate test submission

In [ ]:
test_preds = generate_batch(test_df['prompt_text'].tolist())

submission = pd.DataFrame({
    'ID':         test_df['ID'].values,
    'TargetRLF1': test_preds,
    'TargetR1F1': test_preds,
    'TargetLLM':  test_preds,
})

out_path = SAVE_DIR + 'exp03_mt5small_qlora_r8.csv'
submission.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
submission.head(3)

## Results

| Metric | Baseline (Exp 01) | This run (Exp 03) | Delta |
|--------|-------------------|-------------------|-------|
| ROUGE-1 F1 | *(exp01)* | *(fill)* | — |
| ROUGE-L F1 | *(exp01)* | *(fill)* | — |
| Zindi LB   | *(exp01)* | *(fill)* | — |

**Insight:** *(Fill after running.)*  
**Next:** Experiment 04 — increase LoRA rank to r=16.